# 1. Connecting Python to MySQL

To work with MySQL databases in Python, we need a **connector library** that allows Python to communicate with the MySQL server.

## Available Libraries

| Library | Description | Install Command |
|---------|-------------|------------------|
| `mysql-connector-python` | Official MySQL connector | `pip install mysql-connector-python` |
| `PyMySQL` | Pure Python implementation | `pip install pymysql` |
| `mysqlclient` | Fast C-based connector | `pip install mysqlclient` |

We'll use **mysql-connector-python** as it's the official connector maintained by Oracle.

# 2. Installation

## 2.1 Install MySQL Server

First, you need MySQL Server installed on your machine:
- **Windows**: Download from [MySQL Downloads](https://dev.mysql.com/downloads/mysql/)
- **Mac**: `brew install mysql`
- **Linux**: `sudo apt-get install mysql-server`

## 2.2 Install Python Connector

In [ ]:
# Install mysql-connector-python
# Run this in your terminal or uncomment and run here

# !pip install mysql-connector-python

# 3. Basic Connection

## 3.1 Import and Connect

In [ ]:
import mysql.connector

# Establish connection to MySQL Server
connection = mysql.connector.connect(
    host="localhost",       # Server address (localhost for local machine)
    user="root",            # Your MySQL username
    password="your_password" # Your MySQL password
)

# Check if connection is successful
if connection.is_connected():
    print("Successfully connected to MySQL Server!")
    print(f"MySQL Server version: {connection.get_server_info()}")

# Always close the connection when done
connection.close()
print("Connection closed.")

## 3.2 Connection Parameters Explained

| Parameter | Description | Example |
|-----------|-------------|----------|
| `host` | Server location | `"localhost"` or `"192.168.1.1"` |
| `user` | MySQL username | `"root"` |
| `password` | User's password | `"mypassword"` |
| `database` | Database to use | `"my_database"` |
| `port` | Port number (default 3306) | `3306` |
| `charset` | Character encoding | `"utf8mb4"` |

# 4. Connecting to a Specific Database

In [ ]:
import mysql.connector

# Connect to a specific database
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="ecommerce_db"  # Specify the database
)

if connection.is_connected():
    db_info = connection.get_server_info()
    print(f"Connected to MySQL Server version {db_info}")
    
    # Get the current database
    cursor = connection.cursor()
    cursor.execute("SELECT DATABASE();")
    db_name = cursor.fetchone()
    print(f"Connected to database: {db_name[0]}")

connection.close()

# 5. Creating a Database with Python

In [ ]:
import mysql.connector

# Connect to MySQL (without specifying a database)
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password"
)

# Create a cursor object to execute queries
cursor = connection.cursor()

# Create a new database
cursor.execute("CREATE DATABASE IF NOT EXISTS test_db")
print("Database 'test_db' created successfully!")

# Show all databases
cursor.execute("SHOW DATABASES")
print("\nAvailable databases:")
for db in cursor:
    print(f"  - {db[0]}")

cursor.close()
connection.close()

# 6. The Cursor Object

The **cursor** is the interface for executing SQL queries and retrieving results.

## 6.1 Cursor Methods

| Method | Description |
|--------|-------------|
| `execute(query)` | Execute a single SQL query |
| `executemany(query, data)` | Execute query with multiple data sets |
| `fetchone()` | Fetch one row from result |
| `fetchall()` | Fetch all rows from result |
| `fetchmany(n)` | Fetch n rows from result |
| `close()` | Close the cursor |

In [ ]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="test_db"
)

cursor = connection.cursor()

# ==========================================
# CREATING A TABLE
# ==========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS employees (
    emp_id INT PRIMARY KEY AUTO_INCREMENT,
    name VARCHAR(100) NOT NULL,
    department VARCHAR(50),
    salary DECIMAL(10, 2),
    hire_date DATE
)
"""

cursor.execute(create_table_query)
print("Table 'employees' created successfully!")

# Show tables in the database
cursor.execute("SHOW TABLES")
print("\nTables in database:")
for table in cursor:
    print(f"  - {table[0]}")

cursor.close()
connection.close()

# 7. Error Handling

Always use try-except blocks to handle potential database errors.

In [ ]:
import mysql.connector
from mysql.connector import Error

def create_connection():
    """Create a database connection with error handling."""
    connection = None
    try:
        connection = mysql.connector.connect(
            host="localhost",
            user="root",
            password="your_password",
            database="test_db"
        )
        print("Connection to MySQL DB successful")
    except Error as e:
        print(f"Error: '{e}'")
    
    return connection

# Test the function
conn = create_connection()
if conn:
    conn.close()

# 8. Using Context Manager (Recommended)

The context manager (`with` statement) automatically handles closing connections.

In [ ]:
import mysql.connector
from mysql.connector import Error

# Using context manager - connection closes automatically
try:
    with mysql.connector.connect(
        host="localhost",
        user="root",
        password="your_password",
        database="test_db"
    ) as connection:
        
        with connection.cursor() as cursor:
            cursor.execute("SELECT * FROM employees")
            results = cursor.fetchall()
            
            for row in results:
                print(row)
        
        # Connection automatically commits if no errors
        # Connection automatically closes after the with block

except Error as e:
    print(f"Error: {e}")

print("Connection was automatically closed!")

# 9. Connection Configuration File

For security, store credentials in a separate config file.

## 9.1 Create config.py file

```python
# config.py
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'your_password',
    'database': 'test_db'
}
```

## 9.2 Use the config

In [ ]:
# Example of how to use config file (create config.py first)

# from config import DB_CONFIG
# import mysql.connector

# Using dictionary unpacking
# connection = mysql.connector.connect(**DB_CONFIG)

# Alternative: Use environment variables
import os

# Set these in your environment or .env file
# os.environ['DB_HOST'] = 'localhost'
# os.environ['DB_USER'] = 'root'
# os.environ['DB_PASSWORD'] = 'your_password'
# os.environ['DB_NAME'] = 'test_db'

# Then use them:
# connection = mysql.connector.connect(
#     host=os.environ.get('DB_HOST'),
#     user=os.environ.get('DB_USER'),
#     password=os.environ.get('DB_PASSWORD'),
#     database=os.environ.get('DB_NAME')
# )

print("Config example - uncomment and modify as needed")

# 10. Reusable Database Class

A clean, reusable class for database operations.

In [ ]:
import mysql.connector
from mysql.connector import Error

class MySQLDatabase:
    """A class to handle MySQL database operations."""
    
    def __init__(self, host, user, password, database=None):
        """Initialize database connection parameters."""
        self.host = host
        self.user = user
        self.password = password
        self.database = database
        self.connection = None
    
    def connect(self):
        """Establish connection to MySQL server."""
        try:
            self.connection = mysql.connector.connect(
                host=self.host,
                user=self.user,
                password=self.password,
                database=self.database
            )
            if self.connection.is_connected():
                print(f"Connected to MySQL: {self.database or 'Server'}")
                return True
        except Error as e:
            print(f"Connection Error: {e}")
            return False
    
    def disconnect(self):
        """Close the database connection."""
        if self.connection and self.connection.is_connected():
            self.connection.close()
            print("MySQL connection closed.")
    
    def execute_query(self, query, params=None):
        """Execute a query (INSERT, UPDATE, DELETE, CREATE)."""
        cursor = self.connection.cursor()
        try:
            cursor.execute(query, params)
            self.connection.commit()
            print("Query executed successfully.")
            return cursor.lastrowid  # Returns ID for INSERT
        except Error as e:
            print(f"Query Error: {e}")
            return None
        finally:
            cursor.close()
    
    def fetch_query(self, query, params=None):
        """Execute a SELECT query and return results."""
        cursor = self.connection.cursor(dictionary=True)  # Returns dict instead of tuple
        try:
            cursor.execute(query, params)
            results = cursor.fetchall()
            return results
        except Error as e:
            print(f"Query Error: {e}")
            return None
        finally:
            cursor.close()


# Example usage:
# db = MySQLDatabase('localhost', 'root', 'your_password', 'test_db')
# db.connect()
# results = db.fetch_query("SELECT * FROM employees")
# print(results)
# db.disconnect()

print("MySQLDatabase class defined - ready to use!")

# 11. Summary

## Connection Steps

1. **Install** `mysql-connector-python`
2. **Import** `mysql.connector`
3. **Connect** using `mysql.connector.connect()`
4. **Create cursor** with `connection.cursor()`
5. **Execute queries** with `cursor.execute()`
6. **Commit changes** with `connection.commit()`
7. **Close** cursor and connection

## Best Practices

| Practice | Description |
|----------|-------------|
| **Use try-except** | Always handle potential errors |
| **Use context managers** | Auto-close connections with `with` |
| **Store credentials securely** | Use config files or env variables |
| **Close connections** | Always close when done |
| **Use parameterized queries** | Prevent SQL injection (covered in CRUD) |